In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv('dataset/data_engineered_1.csv')

Финальная проверка целостности

In [9]:
print("=" * 60)
print("финальная проверка датасета")
print("=" * 60)
print(f"Размер: {df.shape}")
print(f"Пропусков: {df.isnull().sum().sum()}")
print(f"Дубликатов: {df.duplicated().sum()}")
print(f"\nТипы данных:")
print(df.dtypes)

# Сброс дубликатов
print(f"До: {df.shape[0]} строк, дубликатов: {df.duplicated().sum()}")
# удаление + сброс индекса
df = df.drop_duplicates().reset_index(drop=True)
print(f"После: {df.shape[0]} строк, дубликатов: {df.duplicated().sum()}")

финальная проверка датасета
Размер: (11113, 14)
Пропусков: 0
Дубликатов: 89

Типы данных:
price               float64
metro_station           str
minutes_to_metro    float64
rooms               float64
area                float64
floor               float64
total_floors          int64
renovation              str
floor_ratio         float64
is_first_floor        int64
is_last_floor         int64
is_studio             int64
log_area            float64
log_price           float64
dtype: object
До: 11113 строк, дубликатов: 89
После: 11024 строк, дубликатов: 0


Разделение на X и y, train/test split

In [10]:
from sklearn.model_selection import train_test_split

# Признаки (всё, кроме целевых)
feature_cols = [
    'area', 'rooms', 'floor', 'total_floors', 'minutes_to_metro',
    'floor_ratio', 'is_first_floor', 'is_last_floor', 'is_studio',
    'log_area',
    'metro_station', 'renovation',
]

X = df[feature_cols].copy()
y = df['price'].copy()           # основная целевая для древесных моделей
y_log = df['log_price'].copy()   # альтернативная для линейной регрессии

# 80/20 split
X_train, X_test, y_train, y_test, y_log_train, y_log_test = train_test_split(
    X, y, y_log,
    test_size=0.2,
    random_state=42,
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}, диапазон [{y_train.min():,.0f}; {y_train.max():,.0f}] ₽")
print(f"y_test:  {y_test.shape}, диапазон [{y_test.min():,.0f}; {y_test.max():,.0f}] ₽")


X_train: (8819, 12)
X_test:  (2205, 12)
y_train: (8819,), диапазон [3,510,000; 459,192,280] ₽
y_test:  (2205,), диапазон [3,550,000; 458,400,000] ₽


Контроль: распределения train и test похожи

In [11]:
# Это обязательная проверка. Если train и test разошлись по ключевым статистикам, модель будет давать смещённые оценки

import pandas as pd

stats = pd.DataFrame({
    'train_mean':   X_train.select_dtypes(include='number').mean(),
    'test_mean':    X_test.select_dtypes(include='number').mean(),
    'train_std':    X_train.select_dtypes(include='number').std(),
    'test_std':     X_test.select_dtypes(include='number').std(),
})
stats['mean_diff_%'] = (stats['test_mean'] - stats['train_mean']) / stats['train_mean'] * 100
print(stats.round(3))

print(f"\nЦена:")
print(f"  train mean: {y_train.mean():>15,.0f} ₽   train median: {y_train.median():>15,.0f} ₽")
print(f"  test  mean: {y_test.mean():>15,.0f} ₽   test  median: {y_test.median():>15,.0f} ₽")


                  train_mean  test_mean  train_std  test_std  mean_diff_%
area                  80.126     81.098     63.684    64.692        1.213
rooms                  2.335      2.356      1.687     1.711        0.911
floor                  8.870      8.549      8.523     8.084       -3.624
total_floors          17.641     17.181     12.355    11.719       -2.604
minutes_to_metro      12.454     12.422      7.181     7.165       -0.258
floor_ratio            0.516      0.515      0.291     0.289       -0.185
is_first_floor         0.094      0.088      0.292     0.283       -6.741
is_last_floor          0.081      0.082      0.273     0.275        1.106
is_studio              0.172      0.171      0.377     0.376       -0.868
log_area               4.136      4.147      0.720     0.720        0.260

Цена:
  train mean:      46,126,877 ₽   train median:      20,000,000 ₽
  test  mean:      48,190,719 ₽   test  median:      20,000,000 ₽


Сохранение

In [12]:
import os
os.makedirs('../compiled_ml_dataset', exist_ok=True)

X_train.to_csv('../compiled_ml_dataset/X_train.csv', index=False)
X_test.to_csv('../compiled_ml_dataset/X_test.csv', index=False)
y_train.to_csv('../compiled_ml_dataset/y_train.csv', index=False, header=True)
y_test.to_csv('../compiled_ml_dataset/y_test.csv', index=False, header=True)
y_log_train.to_csv('../compiled_ml_dataset/y_log_train.csv', index=False, header=True)
y_log_test.to_csv('../compiled_ml_dataset/y_log_test.csv', index=False, header=True)

print("Все выборки сохранены в dataset/")


Все выборки сохранены в dataset/
